# Pandas — Groeperen, aggregeren en samenvoegen

Wanneer je data per categorie wilt samenvatten — bijv. gemiddelde fooi per dag, totale verkoop per product — gebruik je **groupby**. Deze notebook behandelt:

- `.groupby()` met aggregatiefuncties
- `.apply()` met aangepaste functies en **lambda's**
- `.map()` voor waarde-mapping
- **Samenvoegen** van DataFrames met `pd.concat()` en `pd.merge()`

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

tips = sns.load_dataset("tips")
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


## `.groupby()` — Groeperen en aggregeren

De `groupby` werkt in drie stappen: **split** → **apply** → **combine**.

1. Verdeel het DataFrame in groepen op basis van een kolom
2. Pas een aggregatiefunctie toe op elke groep
3. Combineer de resultaten tot één DataFrame

In [2]:
# Mean bill and tip per day
tips.groupby("day")[["total_bill", "tip"]].mean().round(2)

/tmp/ipykernel_1412/979432825.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")[["total_bill", "tip"]].mean().round(2)


,total_bill,tip
day,,
Thur,17.68,2.77
Fri,17.15,2.73
Sat,20.44,2.99
Sun,21.41,3.26


In [3]:
# Multiple aggregations at once with .agg()
tips.groupby("day")["tip"].agg(["mean", "median", "std", "count"]).round(2)

/tmp/ipykernel_1412/3698489606.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("day")["tip"].agg(["mean", "median", "std", "count"]).round(2)


,mean,median,std,count
day,,,,
Thur,2.77,2.30,1.24,62
Fri,2.73,3.00,1.02,19
Sat,2.99,2.75,1.63,87
Sun,3.26,3.15,1.23,76


In [4]:
# Different aggregations per column
tips.groupby("sex").agg(
    mean_bill=("total_bill", "mean"),
    total_tip=("tip", "sum"),
    visit_count=("total_bill", "count"),
).round(2)

/tmp/ipykernel_1412/2316974659.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby("sex").agg(


,mean_bill,total_tip,visit_count
sex,,,
Male,20.74,485.07,157
Female,18.06,246.51,87


In [5]:
# Group by multiple columns
tips.groupby(["sex", "smoker"])["tip"].mean().round(2)

/tmp/ipykernel_1412/3189208785.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tips.groupby(["sex", "smoker"])["tip"].mean().round(2)


sex     smoker
Male    Yes       3.05
        No        3.11
Female  Yes       2.93
        No        2.77
Name: tip, dtype: float64

In [6]:
# reset_index() converts group labels back to regular columns
result = tips.groupby("day")["tip"].mean().reset_index()
result.columns = ["day", "mean_tip"]
result

/tmp/ipykernel_1412/4261992171.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  result = tips.groupby("day")["tip"].mean().reset_index()


,day,mean_tip
0,Thur,2.771452
1,Fri,2.734737
2,Sat,2.993103
3,Sun,3.255132


## `.apply()` — Aangepaste functies

Wanneer een standaard aggregatie niet volstaat, gebruik je `.apply()` met een zelfgeschreven functie of een **lambda**.

In [7]:
# Add a new column: tip as a percentage of the bill
tips["tip_pct"] = tips["tip"] / tips["total_bill"] * 100
tips[["total_bill", "tip", "tip_pct"]].head()

,total_bill,tip,tip_pct
0,16.99,1.01,5.944673
1,10.34,1.66,16.054159
2,21.01,3.50,16.658734
3,23.68,3.31,13.978041
4,24.59,3.61,14.680765


In [8]:
# .apply() with a named function
def categorize_tip(pct):
    """Classify a tip percentage as low, normal or generous."""
    if pct < 10:
        return "low"
    elif pct < 20:
        return "normal"
    else:
        return "generous"


tips["tip_category"] = tips["tip_pct"].apply(categorize_tip)
tips["tip_category"].value_counts()

tip_category
normal      178
generous     39
low          27
Name: count, dtype: int64

In [9]:
# Lambda inline: more compact for simple transformations
tips["bill_rounded"] = tips["total_bill"].apply(lambda x: round(x / 5) * 5)
tips[["total_bill", "bill_rounded"]].head()

,total_bill,bill_rounded
0,16.99,15
1,10.34,10
2,21.01,20
3,23.68,25
4,24.59,25


## `.map()` — Waarden omzetten via een dictionary

In [10]:
# Map abbreviated day names to full English names
day_mapping = {"Thur": "Thursday", "Fri": "Friday", "Sat": "Saturday", "Sun": "Sunday"}
tips["day_name"] = tips["day"].map(day_mapping)
tips[["day", "day_name"]].drop_duplicates()

,day,day_name
0,Sun,Sunday
19,Sat,Saturday
77,Thur,Thursday
90,Fri,Friday


## DataFrames samenvoegen

### `pd.concat()` — Rijen of kolommen toevoegen

In [11]:
# Stack two DataFrames vertically
df1 = pd.DataFrame({"name": ["Alice", "Bob"], "score": [88, 74]})
df2 = pd.DataFrame({"name": ["Charlie", "Diana"], "score": [92, 81]})

combined = pd.concat([df1, df2], ignore_index=True)
print(combined)

      name  score
0    Alice     88
1      Bob     74
2  Charlie     92
3    Diana     81


### `pd.merge()` — Tabellen koppelen op een sleutelkolom

`merge()` is het equivalent van een SQL `JOIN`.

In [12]:
students = pd.DataFrame(
    {
        "student_id": [1, 2, 3, 4],
        "name": ["Alice", "Bob", "Charlie", "Diana"],
        "program_id": [101, 102, 101, 103],
    }
)

programs = pd.DataFrame(
    {
        "program_id": [101, 102, 103],
        "program_name": ["Computer Science", "Electronics", "Mechanical Engineering"],
    }
)

# Inner join: only rows with a match in both tables
result = pd.merge(students, programs, on="program_id", how="inner")
result

,student_id,name,program_id,program_name
0,1,Alice,101,Computer Science
1,2,Bob,102,Electronics
2,3,Charlie,101,Computer Science
3,4,Diana,103,Mechanical Engineering


In [13]:
# Left join: all rows from the left table, NaN where there is no match
extra_student = pd.DataFrame(
    {
        "student_id": [5],
        "name": ["Eve"],
        "program_id": [999],  # does not exist in programs
    }
)
students_extended = pd.concat([students, extra_student], ignore_index=True)

result_left = pd.merge(students_extended, programs, on="program_id", how="left")
result_left

,student_id,name,program_id,program_name
0,1,Alice,101,Computer Science
1,2,Bob,102,Electronics
2,3,Charlie,101,Computer Science
3,4,Diana,103,Mechanical Engineering
4,5,Eve,999,NaN


---

## Oefeningen

Gebruik de **Titanic-dataset**: `sns.load_dataset("titanic")`.

In [14]:
titanic = sns.load_dataset("titanic")

**Oefening 1** — Bereken het **overlevingspercentage** per klasse (`pclass`). (Tip: het gemiddelde van de kolom `survived` geeft het percentage, want 0 = niet overleefd, 1 = overleefd.)

In [15]:
# your solution here

**Oefening 2** — Bereken per combinatie van `sex` en `pclass` de gemiddelde leeftijd, de gemiddelde prijs (`fare`) en het aantal passagiers. Gebruik `.agg()` met benoemde aggregaties.

In [16]:
# your solution here

**Oefening 3** — Voeg een kolom `age_category` toe met de waarden `"child"` (< 18), `"adult"` (18–60) of `"senior"` (> 60). Gebruik `.apply()` met een functie.

In [17]:
df = titanic.copy()
df["age"] = df["age"].fillna(df["age"].median())
# your solution here

**Oefening 4** — Gebruik `.map()` om een kolom `class_name` te maken die `pclass` (1, 2, 3) omzet naar `"First class"`, `"Second class"`, `"Third class"`.

In [18]:
# your solution here

**Oefening 5** — Voeg de Titanic-dataset samen met een fictief DataFrame van havens:
```python
ports = pd.DataFrame({
    "embarked": ["S", "C", "Q"],
    "city":    ["Southampton", "Cherbourg", "Queenstown"],
    "country": ["UK", "France", "Ireland"],
})
```
Gebruik een left join op de kolom `embarked`. Hoeveel rijen hebben een `NaN` voor `city` na de join?

In [19]:
ports = pd.DataFrame(
    {
        "embarked": ["S", "C", "Q"],
        "city": ["Southampton", "Cherbourg", "Queenstown"],
        "country": ["UK", "France", "Ireland"],
    }
)
# your solution here